# 🧹 Data Cleaning Pipeline
## Standardize, Clean, and Prepare Datasets for Analysis

This notebook handles:
- Missing value treatment
- Column name standardization
- Date format normalization
- Outlier detection and handling
- Data type conversions

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Paths
RAW_PATH = Path('../data/raw/')
PROCESSED_PATH = Path('../data/processed/')
PROCESSED_PATH.mkdir(exist_ok=True)

print("Data Cleaning Pipeline Ready")

Data Cleaning Pipeline Ready


## 1. Utility Functions

In [2]:
def standardize_columns(df):
    """Standardize column names: lowercase, replace spaces with underscores."""
    df.columns = (df.columns
                  .str.lower()
                  .str.strip()
                  .str.replace(' ', '_')
                  .str.replace('-', '_')
                  .str.replace('(', '')
                  .str.replace(')', ''))
    return df

def standardize_state_names(df, col='state_name'):
    """Standardize state names to title case and fix common variations."""
    if col not in df.columns:
        return df
    
    df[col] = df[col].str.strip().str.title()
    
    # Common corrections
    replacements = {
        'Andaman And Nicobar Islands': 'Andaman & Nicobar',
        'Jammu And Kashmir': 'Jammu & Kashmir',
        'Dadra And Nagar Haveli': 'Dadra & Nagar Haveli',
        'Daman And Diu': 'Daman & Diu'
    }
    df[col] = df[col].replace(replacements)
    return df

def parse_dates(df, date_col='date'):
    """Parse dates with multiple format handling."""
    if date_col not in df.columns:
        return df
    
    # Try multiple formats
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce', infer_datetime_format=True)
    return df

def handle_missing_numeric(df, strategy='median', columns=None):
    """Handle missing values in numeric columns."""
    if columns is None:
        columns = df.select_dtypes(include=[np.number]).columns
    
    for col in columns:
        if col in df.columns:
            if strategy == 'median':
                df[col] = df[col].fillna(df[col].median())
            elif strategy == 'mean':
                df[col] = df[col].fillna(df[col].mean())
            elif strategy == 'zero':
                df[col] = df[col].fillna(0)
            elif strategy == 'ffill':
                df[col] = df[col].ffill()
    return df

def remove_outliers(df, columns, method='iqr', threshold=1.5):
    """Remove outliers using IQR or z-score method."""
    df_clean = df.copy()
    
    for col in columns:
        if col not in df.columns:
            continue
            
        if method == 'iqr':
            Q1 = df_clean[col].quantile(0.25)
            Q3 = df_clean[col].quantile(0.75)
            IQR = Q3 - Q1
            mask = (df_clean[col] >= Q1 - threshold * IQR) & (df_clean[col] <= Q3 + threshold * IQR)
            df_clean = df_clean[mask]
        elif method == 'zscore':
            z_scores = np.abs((df_clean[col] - df_clean[col].mean()) / df_clean[col].std())
            df_clean = df_clean[z_scores < threshold]
    
    return df_clean

def cleaning_report(df_before, df_after, name):
    """Generate a cleaning report."""
    print(f"\n{'='*50}")
    print(f"📋 Cleaning Report: {name}")
    print(f"{'='*50}")
    print(f"Rows: {len(df_before):,} → {len(df_after):,} ({len(df_before)-len(df_after):,} removed)")
    print(f"Columns: {len(df_before.columns)} → {len(df_after.columns)}")
    print(f"Missing values before: {df_before.isnull().sum().sum():,}")
    print(f"Missing values after: {df_after.isnull().sum().sum():,}")

---
## 2. Clean Key Datasets

### 2.1 Minimum Support Prices

In [3]:
# Load
msp_raw = pd.read_csv(RAW_PATH / 'minimum-support-prices.csv')
print(f"Loaded MSP: {len(msp_raw):,} rows")

# Clean
msp = msp_raw.copy()
msp = standardize_columns(msp)
msp['crop'] = msp['crop'].str.strip()
msp['season'] = msp['season'].str.strip()

# Handle single missing value
msp = handle_missing_numeric(msp, strategy='ffill', columns=['min_support_price'])

# Extract year range for easier filtering
msp['year_start'] = msp['year'].str.split('-').str[0].astype(int)

# Save
msp.to_csv(PROCESSED_PATH / 'msp_cleaned.csv', index=False)
cleaning_report(msp_raw, msp, 'Minimum Support Prices')
msp.head()

Loaded MSP: 737 rows

📋 Cleaning Report: Minimum Support Prices
Rows: 737 → 737 (0 removed)
Columns: 5 → 6
Missing values before: 1
Missing values after: 0


,id,year,crop,season,min_support_price,year_start
0,0,2022-2023,Paddy - Common,Kharif,2040.0,2022
1,1,2022-2023,Paddy - Grade 'A',Kharif,2060.0,2022
2,2,2022-2023,Jowar - Hybrid,Kharif,2970.0,2022
3,3,2022-2023,Jowar - Maldandi,Kharif,2990.0,2022
4,4,2022-2023,Bajra,Kharif,2350.0,2022


### 2.2 Cost of Cultivation

In [4]:
# Load
coc_raw = pd.read_csv(RAW_PATH / 'cost-of-cultivation.csv')
print(f"Loaded Cost of Cultivation: {len(coc_raw):,} rows")

# Clean
coc = coc_raw.copy()
coc = standardize_columns(coc)
coc = standardize_state_names(coc)

# Drop columns with >90% missing
missing_pct = coc.isnull().sum() / len(coc) * 100
cols_to_drop = missing_pct[missing_pct > 90].index.tolist()
print(f"Dropping columns with >90% missing: {cols_to_drop}")
coc = coc.drop(columns=cols_to_drop, errors='ignore')

# Extract year
coc['year_start'] = coc['year'].str.split('-').str[0].astype(int)

# Calculate key metrics
coc['profit_margin_pct'] = ((coc['main_product_value'] - coc['cul_cost_c2']) / 
                            coc['main_product_value'] * 100).round(2)
coc['cost_per_quintal'] = (coc['cul_cost_c2'] / coc['derived_yield']).round(2)

# Save
coc.to_csv(PROCESSED_PATH / 'cost_of_cultivation_cleaned.csv', index=False)
cleaning_report(coc_raw, coc, 'Cost of Cultivation')
coc.head()

Loaded Cost of Cultivation: 3,880 rows
Dropping columns with >90% missing: ['opr_cost_contractor_pay']

📋 Cleaning Report: Cost of Cultivation
Rows: 3,880 → 3,880 (0 removed)
Columns: 66 → 68
Missing values before: 13,278
Missing values after: 9,535


,id,year,state_name,state_code,crop_name,crop_code,crop_type,cul_cost_a1,cul_cost_a2,cul_cost_b1,...,fix_cost_owned_land_rental,fix_cost_leased_land_rent,fix_cost_land_rev_tax_cess,fix_cost_depr_impl_farm_build,fix_cost_interest_fix_cap,fix_cost,opr_cost_crop_insurance,year_start,profit_margin_pct,cost_per_quintal
0,0,2018-19,Andhra Pradesh,28,Tur (Arhar),202,Pulses,25062.60,25062.60,26029.29,...,10381.33,0.00,0.00,105.13,966.70,11453.16,0.0,2018,-20.43,5575.56
1,1,2018-19,Bihar,10,Tur (Arhar),202,Pulses,14717.68,14717.68,15477.71,...,15586.67,0.00,240.00,418.30,760.03,17005.00,0.0,2018,10.97,3562.36
2,2,2018-19,Chhattisgarh,22,Tur (Arhar),202,Pulses,14183.54,14183.54,14528.70,...,6488.38,0.00,1.64,336.36,345.17,7171.55,0.0,2018,-11.92,5134.89
3,3,2018-19,Gujarat,24,Tur (Arhar),202,Pulses,27435.91,28655.86,29853.94,...,6988.38,1219.96,17.95,298.56,2418.03,10942.88,0.0,2018,-18.95,6481.33
4,4,2018-19,Karnataka,29,Tur (Arhar),202,Pulses,25298.57,25298.57,26711.34,...,9677.40,0.00,10.82,241.81,1412.77,11342.80,0.0,2018,-7.59,5530.98


### 2.3 Daily Retail Prices

In [5]:
# Load in chunks for large file
print("Loading retail prices (this may take a moment)...")
retail_raw = pd.read_csv(RAW_PATH / 'daily-retail-prices-of-essential-commodities.csv')
print(f"Loaded Retail Prices: {len(retail_raw):,} rows")

# Clean
retail = retail_raw.copy()
retail = standardize_columns(retail)
retail = standardize_state_names(retail)
retail = parse_dates(retail, 'date')

# Standardize commodity names
retail['commodity'] = retail['commodity'].str.strip().str.title()

# Handle missing prices - fill with state-commodity median
retail['price'] = retail.groupby(['state_name', 'commodity'])['price'].transform(
    lambda x: x.fillna(x.median())
)

# Add temporal features
retail['year'] = retail['date'].dt.year
retail['month'] = retail['date'].dt.month
retail['day_of_week'] = retail['date'].dt.dayofweek

# Remove extreme outliers (prices > 10x median or < 0)
median_prices = retail.groupby('commodity')['price'].transform('median')
retail = retail[(retail['price'] > 0) & (retail['price'] < median_prices * 10)]

# Save
retail.to_csv(PROCESSED_PATH / 'retail_prices_cleaned.csv', index=False)
cleaning_report(retail_raw, retail, 'Daily Retail Prices')
retail.head()

Loading retail prices (this may take a moment)...
Loaded Retail Prices: 2,117,473 rows

📋 Cleaning Report: Daily Retail Prices
Rows: 2,117,473 → 2,104,788 (12,685 removed)
Columns: 6 → 9
Missing values before: 79,821
Missing values after: 0


,id,date,state_name,state_code,commodity,price,year,month,day_of_week
0,0,2015-01-01,Andhra Pradesh,28,Rice,26.00,2015,1,3
1,1,2015-01-01,Assam,18,Rice,24.00,2015,1,3
2,2,2015-01-01,Bihar,10,Rice,31.33,2015,1,3
3,3,2015-01-01,Chandigarh,4,Rice,29.00,2015,1,3
4,4,2015-01-01,Chhattisgarh,22,Rice,26.00,2015,1,3


### 2.4 State-level Rainfall

In [6]:
# Load
rainfall_raw = pd.read_csv(RAW_PATH / 'daily-rainfall-at-state-level.csv')
print(f"Loaded State Rainfall: {len(rainfall_raw):,} rows")

# Clean
rainfall = rainfall_raw.copy()
rainfall = standardize_columns(rainfall)
rainfall = standardize_state_names(rainfall)
rainfall = parse_dates(rainfall, 'date')

# Calculate deviation where missing (deviation = (actual - normal) / normal * 100)
mask = rainfall['deviation'].isna() & rainfall['normal'].notna() & (rainfall['normal'] > 0)
rainfall.loc[mask, 'deviation'] = (
    (rainfall.loc[mask, 'actual'] - rainfall.loc[mask, 'normal']) / 
    rainfall.loc[mask, 'normal'] * 100
)

# Fill remaining missing actual rainfall with 0 (no rain)
rainfall['actual'] = rainfall['actual'].fillna(0)

# Add temporal features
rainfall['year'] = rainfall['date'].dt.year
rainfall['month'] = rainfall['date'].dt.month
rainfall['monsoon'] = rainfall['month'].apply(
    lambda x: 'Monsoon' if x in [6,7,8,9] else 'Non-Monsoon'
)

# Save
rainfall.to_csv(PROCESSED_PATH / 'rainfall_state_cleaned.csv', index=False)
cleaning_report(rainfall_raw, rainfall, 'State Rainfall')
rainfall.head()

Loaded State Rainfall: 204,876 rows

📋 Cleaning Report: State Rainfall
Rows: 204,876 → 204,876 (0 removed)
Columns: 8 → 11
Missing values before: 65,566
Missing values after: 48,404


,id,date,state_code,state_name,actual,rfs,normal,deviation,year,month,monsoon
0,0,2009-01-01,5,Uttarakhand,0.0,0.003906,2.19,-100.0,2009,1,Non-Monsoon
1,1,2009-01-01,18,Assam,0.0,0.000000,0.52,-100.0,2009,1,Non-Monsoon
2,2,2009-01-01,16,Tripura,0.0,0.000000,0.09,-100.0,2009,1,Non-Monsoon
3,3,2009-01-01,36,Telangana,0.0,0.000000,0.17,-100.0,2009,1,Non-Monsoon
4,4,2009-01-01,2,Himachal Pradesh,0.0,0.008566,3.31,-100.0,2009,1,Non-Monsoon


### 2.5 Groundwater Levels

In [7]:
# Load
print("Loading groundwater data (large file)...")
gw_raw = pd.read_csv(RAW_PATH / 'cgwb-changes-in-depth-to-water-level.csv')
print(f"Loaded Groundwater: {len(gw_raw):,} rows")

# Clean
gw = gw_raw.copy()
gw = standardize_columns(gw)
gw = standardize_state_names(gw)
gw = parse_dates(gw, 'date')

# Add temporal features
gw['year'] = gw['date'].dt.year
gw['month'] = gw['date'].dt.month
gw['season'] = gw['month'].apply(
    lambda x: 'Pre-Monsoon' if x in [4,5] else 'Post-Monsoon' if x in [10,11] else 'Other'
)

# Remove invalid readings
gw = gw[gw['currentlevel'] >= 0]
gw = gw[gw['currentlevel'] < 500]  # Max reasonable depth

# Save
gw.to_csv(PROCESSED_PATH / 'groundwater_cleaned.csv', index=False)
cleaning_report(gw_raw, gw, 'Groundwater Levels')
gw.head()

Loading groundwater data (large file)...
Loaded Groundwater: 550,850 rows

📋 Cleaning Report: Groundwater Levels
Rows: 550,850 → 550,850 (0 removed)
Columns: 14 → 17
Missing values before: 0
Missing values after: 0


,id,date,state_name,state_code,district_name,district_code,station_name,latitude,longitude,basin,sub_basin,source,currentlevel,level_diff,year,month,season
0,0,2013-11-04,Andaman & Nicobar,35,North And Middle Andaman,632,Laxmipur,13.28556,93.00306,Drainage Area Of Andaman And Nicobar Islands B...,Drainage Area Of Andaman And Nicobar Islands,CGWB,0.10,-1.03,2013,11,Post-Monsoon
1,1,2014-05-14,Andaman & Nicobar,35,North And Middle Andaman,632,Laxmipur,13.28556,93.00306,Drainage Area Of Andaman And Nicobar Islands B...,Drainage Area Of Andaman And Nicobar Islands,CGWB,2.60,2.50,2014,5,Pre-Monsoon
2,2,2014-11-04,Andaman & Nicobar,35,North And Middle Andaman,632,Laxmipur,13.28556,93.00306,Drainage Area Of Andaman And Nicobar Islands B...,Drainage Area Of Andaman And Nicobar Islands,CGWB,0.35,-2.25,2014,11,Post-Monsoon
3,3,2015-05-14,Andaman & Nicobar,35,North And Middle Andaman,632,Laxmipur,13.28556,93.00306,Drainage Area Of Andaman And Nicobar Islands B...,Drainage Area Of Andaman And Nicobar Islands,CGWB,2.52,2.17,2015,5,Pre-Monsoon
4,4,2015-11-04,Andaman & Nicobar,35,North And Middle Andaman,632,Laxmipur,13.28556,93.00306,Drainage Area Of Andaman And Nicobar Islands B...,Drainage Area Of Andaman And Nicobar Islands,CGWB,0.69,-1.83,2015,11,Post-Monsoon


### 2.6 Climate Vulnerability Indicators

In [8]:
# Load state-level
vuln_state_raw = pd.read_csv(RAW_PATH / 'climate-vulnerability-indicators-state-wise.csv')
print(f"Loaded State Vulnerability: {len(vuln_state_raw):,} rows")

# Clean
vuln = vuln_state_raw.copy()
vuln = standardize_columns(vuln)
vuln = standardize_state_names(vuln)

# Keep only key indicators with good data coverage
key_indicators = [
    'state_name', 'state_code', 'climate_vul_in', 'yield_variability',
    'irrigated_area', 'rainfed_agriculture', 'poverty_rate',
    'women_workforce', 'employed_under_mgnrega', 'road_rail_density'
]
vuln_clean = vuln[[col for col in key_indicators if col in vuln.columns]].copy()

# Drop rows with missing vulnerability index
vuln_clean = vuln_clean.dropna(subset=['climate_vul_in'])

# Normalize vulnerability index to 0-1 scale
vuln_clean['vulnerability_normalized'] = (
    (vuln_clean['climate_vul_in'] - vuln_clean['climate_vul_in'].min()) / 
    (vuln_clean['climate_vul_in'].max() - vuln_clean['climate_vul_in'].min())
)

# Risk category
vuln_clean['risk_category'] = pd.cut(
    vuln_clean['vulnerability_normalized'],
    bins=[0, 0.33, 0.67, 1.0],
    labels=['Low', 'Medium', 'High']
)

# Save
vuln_clean.to_csv(PROCESSED_PATH / 'vulnerability_state_cleaned.csv', index=False)
cleaning_report(vuln_state_raw, vuln_clean, 'Climate Vulnerability')
vuln_clean.head(10)

Loaded State Vulnerability: 36 rows

📋 Cleaning Report: Climate Vulnerability
Rows: 36 → 30 (6 removed)
Columns: 29 → 12
Missing values before: 150
Missing values after: 1


,state_name,state_code,climate_vul_in,yield_variability,irrigated_area,rainfed_agriculture,poverty_rate,women_workforce,employed_under_mgnrega,road_rail_density,vulnerability_normalized,risk_category
1,Andhra Pradesh,28,0.510,0.08,46.94,0.53,9.20,36.16,47.0,1.11,0.356863,Medium
2,Arunachal Pradesh,12,0.594,0.18,24.89,0.75,34.67,35.44,14.0,0.51,0.686275,High
3,Assam,18,0.620,0.15,10.47,0.90,31.98,22.46,22.0,4.31,0.788235,High
4,Bihar,10,0.614,0.20,56.59,0.43,33.74,19.07,34.0,2.23,0.764706,High
6,Chhattisgarh,22,0.623,0.16,31.32,0.69,39.93,39.70,32.0,0.72,0.800000,High
8,Goa,30,0.434,0.08,30.23,0.70,5.09,21.92,23.0,6.09,0.058824,Low
9,Gujarat,24,0.562,0.13,41.09,0.59,16.63,23.38,35.0,0.92,0.560784,Medium
10,Haryana,6,0.463,0.08,84.44,0.16,11.16,17.79,28.0,1.93,0.172549,Low
11,Himachal Pradesh,2,0.486,0.10,20.55,0.79,8.06,44.82,42.0,1.16,0.262745,Low
12,Jammu & Kashmir,1,0.550,0.10,43.67,0.56,10.35,19.11,36.0,0.29,0.513725,Medium


---
## 3. Summary of Cleaned Datasets

In [9]:
# List all cleaned files
cleaned_files = list(PROCESSED_PATH.glob('*.csv'))

summary = []
for f in cleaned_files:
    df = pd.read_csv(f, nrows=1000)
    full_df = pd.read_csv(f)
    summary.append({
        'File': f.name,
        'Rows': len(full_df),
        'Columns': len(full_df.columns),
        'Size (MB)': round(f.stat().st_size / 1024 / 1024, 2)
    })

summary_df = pd.DataFrame(summary)
print("\n📁 CLEANED DATASETS READY FOR ANALYSIS")
print(f"Location: {PROCESSED_PATH.absolute()}")
display(summary_df)


📁 CLEANED DATASETS READY FOR ANALYSIS
Location: c:\Users\Asus\OneDrive\Documents\Project-2\notebooks\..\data\processed


,File,Rows,Columns,Size (MB)
0,cost_of_cultivation_cleaned.csv,3880,68,1.67
1,groundwater_cleaned.csv,550850,17,74.28
2,msp_cleaned.csv,737,6,0.03
3,rainfall_state_cleaned.csv,204876,11,14.16
4,retail_prices_cleaned.csv,2104788,9,119.95
5,vulnerability_state_cleaned.csv,30,12,0.00


---
## Next Steps

Proceed to `03_integration.ipynb` to:
1. Merge datasets on common keys (state, district, date, crop)
2. Create unified analysis datasets
3. Build the master district-level dataset

In [10]:
print("\n✅ Data Cleaning Complete!")
print(f"Cleaned files saved to: {PROCESSED_PATH.absolute()}")


✅ Data Cleaning Complete!
Cleaned files saved to: c:\Users\Asus\OneDrive\Documents\Project-2\notebooks\..\data\processed
